In [35]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

In [36]:
file_path = "Student Mental health.csv"
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "shariful07/student-mental-health",
    file_path
)

/var/folders/qp/jxy8rmnj7vj_yrr93rx2bpvm0000gn/T/ipykernel_5645/422904663.py:2: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


In [37]:
df.shape

(101, 11)

In [38]:
df.head()

,Timestamp,Choose your gender,Age,What is your course?,Your current year of Study,What is your CGPA?,Marital status,Do you have Depression?,Do you have Anxiety?,Do you have Panic attack?,Did you seek any specialist for a treatment?
0,8/7/2020 12:02,Female,18.0,Engineering,year 1,3.00 - 3.49,No,Yes,No,Yes,No
1,8/7/2020 12:04,Male,21.0,Islamic education,year 2,3.00 - 3.49,No,No,Yes,No,No
2,8/7/2020 12:05,Male,19.0,BIT,Year 1,3.00 - 3.49,No,Yes,Yes,Yes,No
3,8/7/2020 12:06,Female,22.0,Laws,year 3,3.00 - 3.49,Yes,Yes,No,No,No
4,8/7/2020 12:13,Male,23.0,Mathemathics,year 4,3.00 - 3.49,No,No,No,No,No


In [39]:
df.columns = [
    'timestamp',
    'gender',
    'age',
    'course',
    'year_of_study',
    'cgpa',
    'marital_status',
    'depression',
    'anxiety',
    'panic_attack',
    'treatment'
]

In [40]:
df.head()

,timestamp,gender,age,course,year_of_study,cgpa,marital_status,depression,anxiety,panic_attack,treatment
0,8/7/2020 12:02,Female,18.0,Engineering,year 1,3.00 - 3.49,No,Yes,No,Yes,No
1,8/7/2020 12:04,Male,21.0,Islamic education,year 2,3.00 - 3.49,No,No,Yes,No,No
2,8/7/2020 12:05,Male,19.0,BIT,Year 1,3.00 - 3.49,No,Yes,Yes,Yes,No
3,8/7/2020 12:06,Female,22.0,Laws,year 3,3.00 - 3.49,Yes,Yes,No,No,No
4,8/7/2020 12:13,Male,23.0,Mathemathics,year 4,3.00 - 3.49,No,No,No,No,No


In [41]:
# remove timestamp column
df = df.drop('timestamp', axis=1)

In [42]:
df.head()

,gender,age,course,year_of_study,cgpa,marital_status,depression,anxiety,panic_attack,treatment
0,Female,18.0,Engineering,year 1,3.00 - 3.49,No,Yes,No,Yes,No
1,Male,21.0,Islamic education,year 2,3.00 - 3.49,No,No,Yes,No,No
2,Male,19.0,BIT,Year 1,3.00 - 3.49,No,Yes,Yes,Yes,No
3,Female,22.0,Laws,year 3,3.00 - 3.49,Yes,Yes,No,No,No
4,Male,23.0,Mathemathics,year 4,3.00 - 3.49,No,No,No,No,No


In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   gender          101 non-null    str    
 1   age             100 non-null    float64
 2   course          101 non-null    str    
 3   year_of_study   101 non-null    str    
 4   cgpa            101 non-null    str    
 5   marital_status  101 non-null    str    
 6   depression      101 non-null    str    
 7   anxiety         101 non-null    str    
 8   panic_attack    101 non-null    str    
 9   treatment       101 non-null    str    
dtypes: float64(1), str(9)
memory usage: 8.0 KB


In [44]:
# What is the prevalence of mental health conditions?
# Convert responses to numeric 0/1 where applicable and compute mean as prevalence
cols = ['depression', 'anxiety', 'panic_attack']

# Map common string/boolean representations to binary 1/0
df[cols] = df[cols].replace({
    'Yes': 1, 'No': 0, 'yes': 1, 'no': 0, 'Y': 1, 'N': 0, 'y': 1, 'n': 0,
    'True': 1, 'False': 0, 'true': 1, 'false': 0,
    '1': 1, '0': 0
})

# Coerce any remaining values to numeric (non-convertible -> NaN)
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')

# Calculate prevalence (mean of 0/1 gives proportion)
prevalence = pd.DataFrame({
    'metric': cols,
    'rate': df[cols].mean().values
})

prevalence


,metric,rate
0,depression,0.346535
1,anxiety,0.336634
2,panic_attack,0.326733


In [45]:
#Is anxiety related to depression?
anxiety_vs_depression = pd.crosstab(
    df["depression"],
    df["anxiety"],
    normalize="index"
)

In [46]:
anxiety_vs_depression

anxiety,0,1
depression,,
0,0.757576,0.242424
1,0.485714,0.514286


In [47]:
#Do panic attacks increase depression risk?
panic_vs_depression = pd.crosstab(
    df["depression"],
    df["panic_attack"],
    normalize="index"
)
panic_vs_depression

panic_attack,0,1
depression,,
0,0.757576,0.242424
1,0.514286,0.485714


In [48]:
#Does gender affect depression?
gender_depression = (
    df.groupby("gender")["depression"]
    .mean()
    .rename("depression_rate")
    .reset_index()
)
gender_depression

,gender,depression_rate
0,Female,0.386667
1,Male,0.230769


In [49]:
#Does year of study influence anxiety?
year_anxiety = (
    df.groupby("year_of_study")["anxiety"]
    .mean()
    .rename("anxiety_rate")
    .reset_index()
)
year_anxiety

,year_of_study,anxiety_rate
0,Year 1,0.500000
1,Year 2,0.437500
2,Year 3,0.368421
3,year 1,0.317073
4,year 2,0.300000
5,year 3,0.200000
6,year 4,0.250000
